# Spin and inclination from an image: a CNN, and how much blur and noise it takes to fail (GPU)

The inverse problem. Given an image of a black hole's accretion disk, as a telescope
would see it, read off the spin and the viewing inclination. The images come from
the kerr-lensing dataset (128x128): each black hole's ray-traced geometry with a
random emission profile pushed through it, including the g^4 Doppler and
gravitational shift. A fresh profile every epoch means the CNN has to read the
black hole, not memorise a disk pattern.

Then the robustness study: blur the test images with a telescope beam (measured
as the beam's width at half maximum over the shadow's diameter; the Event Horizon
Telescope's image of M87* is about 0.5) and add noise, and find how much it takes
before the estimates are no better than half of guessing. Two CNNs: one trained
on clean images, one trained on randomly blurred and noisy ones.

**Before running:** Settings -> Accelerator -> **GPU T4**, Internet **on**, and
**Add Data -> kerr-lensing**.

In [ ]:
# The repo goes to /tmp, so the notebook's output holds only the results.
import os, shutil, sys
REPO_URL = "https://github.com/amark-23/black-hole-ml.git"
REPO = "/tmp/black-hole-ml"
if os.path.exists(REPO):
    shutil.rmtree(REPO)
!git clone -q --depth 1 {REPO_URL} {REPO}
!pip -q install -e {REPO}
for sub in ("ml", "ml/datagen", "simulation/viz"):     # bhml, the generator, the ray tracer
    sys.path.insert(0, os.path.join(REPO, sub))

import glob, json, time
import numpy as np
import torch
import lensing                                   # ml/datagen/lensing.py
from bhml import cnn_inverse as C
from bhml.lensing_data import load_lensing, fixed_emissions
from bhml.models import count_params

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device, "|", torch.cuda.get_device_name(0) if device == "cuda" else "CPU (enable the GPU)")
OUT = "/kaggle/working/cnn_results"
os.makedirs(OUT, exist_ok=True)

## Settings

In [ ]:
CFG = dict(C.DEFAULTS)          # width 32, 5 stages, 120 epochs, batch 64, AdamW 1e-3, ...
CFG.update(n_holes=2000, seed=0, res=128)
BLURS = [0, 0.05, 0.1, 0.2, 0.3, 0.5, 0.75, 1.0]    # beam FWHM / shadow diameter (EHT M87*: 0.5)
NOISES = [0, 0.01, 0.02, 0.05, 0.1, 0.2, 0.3, 0.5]  # noise std / peak brightness
DATA_IN = "/kaggle/input"                   # where Add Data mounts the published set
DATA_WORK = "/tmp/kerr_lensing"             # traced here if it is not attached

## Data

The published kerr-lensing set at 128x128: 1,600 black holes to train on, 200 to
validate, 200 to test.

In [ ]:
def published(res):
    for m in glob.glob(f"{DATA_IN}/**/manifest.json", recursive=True):
        man = json.load(open(m))
        if man.get("name") == "kerr_lensing" and man.get("res") == res:
            return os.path.dirname(m)

src = published(CFG["res"])
if src is None:
    src = f"{DATA_WORK}_{CFG['res']}"
    if not os.path.exists(os.path.join(src, "manifest.json")):
        print("kerr-lensing not attached: tracing it here")
        lensing.generate(src, n=CFG["n_holes"], res=CFG["res"], seed=CFG["seed"], device=device,
                         progress=lambda s: None)
full = load_lensing(src)
train, val, test = full.subset("train"), full.subset("val"), full.subset("test")
unfinished = float((full.otype == 0).mean())
print(f"train {len(train)}, val {len(val)}, test {len(test)} holes at {full.res}x{full.res}; "
      f"{100 * unfinished:.3f}% unfinished pixels")
if unfinished > 0.0015:
    print("WARNING: this looks like the kerr-lensing version made before the ray-tracer fix; "
          "regenerate it with dataset_lensing_kaggle.ipynb and attach the new version.")
emis = fixed_emissions(CFG["n_holes"])      # one fixed profile per hole, for val and test
baseline = C.guess_baseline(train, test)
print(f"guessing the mean: spin error {baseline['spin']:.3f}, inclination error {baseline['incl']:.1f} deg")

## What the network sees

One test image through the blur ladder (top) and the noise ladder (bottom), after
the asinh stretch the network gets.

In [ ]:
row = int(np.argsort(test.params[:, 1])[120])
C.plot_degradations(test, row, emis, BLURS[::2] + [1.0], NOISES[::2] + [0.5], f"{OUT}/degradations.png", device)
from IPython.display import Image, display
display(Image(f"{OUT}/degradations.png"))

## Train

Same architecture, data, loss and schedule for both; the degraded model sees each
training image blurred (up to 1.0) and/or noisy (up to 0.3) with probability 0.6
each. Each keeps its best validation epoch, measured on its own kind of images.

In [ ]:
models, hists = {}, {}
for name, aug in (("clean training", False), ("degraded training", True)):
    models[name], hists[name] = C.train_model(train, val, emis, CFG, device, augment=aug)
print({k: f"{count_params(m) / 1e6:.2f}M" for k, m in models.items()})

## Accuracy on clean test images

In [ ]:
clean = {name: C.predict(m, test, emis, device) for name, m in models.items()}
print(f"{'model':20s} {'spin error':>11s} {'inclination error':>18s}")
for name, (p, t) in clean.items():
    e = C.maes(p, t)
    print(f"{name:20s} {e['spin']:11.3f} {e['incl']:16.2f}°")
print(f"{'guessing the mean':20s} {baseline['spin']:11.3f} {baseline['incl']:16.2f}°")
p, t = clean["clean training"]
inc = t[:, 1]
for lo, hi in ((15, 35), (35, 55), (55, 75), (75, 85)):
    s = (inc >= lo) & (inc < hi + (hi == 85))
    e = C.maes(p[s], t[s])
    print(f"  clean model, inclination {lo}-{hi}°: spin error {e['spin']:.3f}, inclination error {e['incl']:.2f}° (n={s.sum()})")
C.plot_scatter(clean, f"{OUT}/scatter.png")
display(Image(f"{OUT}/scatter.png"))

## Robustness: how much blur and noise before it fails

"Fails" here means the error reaches half of what guessing the mean costs.

In [ ]:
sweeps = {name: C.sweep(m, test, emis, device, BLURS, NOISES) for name, m in models.items()}
for name, sw in sweeps.items():
    print(name)
    for kind in ("blur", "noise"):
        rows = "  ".join(f"{lvl:g}: {m['spin']:.3f}/{m['incl']:.1f}°" for lvl, m in sw[kind])
        print(f"  {kind:5s} (spin/incl)  {rows}")
    for kind in ("blur", "noise"):
        for key in ("spin", "incl"):
            f = C.failure_level(sw[kind], key, baseline)
            print(f"  {key:4s} fails at {kind} = " + (f"{f:.2f}" if f is not None else f"> {sw[kind][-1][0]:g} (never within the sweep)"))
C.plot_robustness(sweeps, baseline, f"{OUT}/robustness.png")
display(Image(f"{OUT}/robustness.png"))

In [ ]:
results = dict(cfg=CFG, blurs=BLURS, noises=NOISES, baseline=baseline,
               clean={k: C.maes(*v) for k, v in clean.items()}, sweeps=sweeps,
               failure={name: {f"{kind}_{key}": C.failure_level(sw[kind], key, baseline)
                               for kind in ("blur", "noise") for key in ("spin", "incl")}
                        for name, sw in sweeps.items()},
               histories=hists, params={k: count_params(m) for k, m in models.items()},
               n_train=len(train), n_val=len(val), n_test=len(test))
json.dump(results, open(f"{OUT}/results.json", "w"), indent=2)
for name, m in models.items():
    torch.save(m.state_dict(), f"{OUT}/cnn_{name.split()[0]}.pt")
print("saved to", OUT, ":", sorted(os.listdir(OUT)))

## Notes

- **What the CNN can read.** The disk's inner edge sits at the innermost stable
  orbit, which moves from 6 M to about 1.5 M as the spin rises; the shadow's size
  and asymmetry change with spin; and the Doppler-bright side changes with both
  spin and inclination. The inner-edge cue is how spins are measured from real
  accretion disks ("continuum fitting").
- **Brightness is relative.** Each image is scaled by its own bright level (the
  99.5th percentile), since a real observation does not tell you the source's
  luminosity.
- **Blur** is a Gaussian beam of the given width, **noise** is Gaussian with a
  standard deviation of the given fraction of the blurred image's peak.